# Run All PCA Notebooks

Executes every PCA notebook in this folder (excluding this runner notebook).

In [ ]:
from __future__ import annotations

import os
import time
from contextlib import contextmanager
from pathlib import Path

import nbformat
import pandas as pd
from nbclient import NotebookClient
from tqdm.auto import tqdm

In [ ]:
def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not locate repository root (missing pyproject.toml).")


@contextmanager
def pushd(target_dir: Path):
    prev = Path.cwd()
    os.chdir(target_dir)
    try:
        yield
    finally:
        os.chdir(prev)


REPO_ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOK_DIR = REPO_ROOT / "feature_selection" / "pca" / "select_pca"
RUNNER_NAME = "run_all_pca.ipynb"
EXECUTE_TIMEOUT = None
KERNEL_NAME = "python3"
CONTINUE_ON_ERROR = True

targets = sorted(
    p for p in NOTEBOOK_DIR.glob("*.ipynb")
    if p.name != RUNNER_NAME
)

print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Runner: {RUNNER_NAME}")
print(f"Targets: {len(targets)}")
for p in targets:
    print(f" - {p.name}")

In [ ]:
results: list[dict[str, object]] = []

for notebook_path in tqdm(targets, total=len(targets), desc="Running PCA notebooks"):
    row = {"notebook": notebook_path.name, "status": "pending", "seconds": None, "error": None}
    started = time.perf_counter()

    try:
        nb = nbformat.read(notebook_path, as_version=4)
        with pushd(NOTEBOOK_DIR):
            client = NotebookClient(
                nb,
                timeout=EXECUTE_TIMEOUT,
                kernel_name=KERNEL_NAME,
            )
            client.execute()
        row["status"] = "ok"
    except Exception as exc:  # noqa: BLE001
        row["status"] = "failed"
        row["error"] = f"{type(exc).__name__}: {exc}"
        if not CONTINUE_ON_ERROR:
            row["seconds"] = round(time.perf_counter() - started, 3)
            results.append(row)
            raise
    finally:
        if row["seconds"] is None:
            row["seconds"] = round(time.perf_counter() - started, 3)

    results.append(row)

results_df = pd.DataFrame(results).sort_values(["status", "notebook"]).reset_index(drop=True)
display(results_df)

ok_count = int((results_df["status"] == "ok").sum()) if not results_df.empty else 0
failed_count = int((results_df["status"] == "failed").sum()) if not results_df.empty else 0
print(f"Completed. ok={ok_count} failed={failed_count}")